# Week 7: Model Evaluation

**CS 203 -- Software Tools and Techniques for AI**

Prof. Nipun Batra, IIT Gandhinagar

**Goal**: Learn how to correctly evaluate machine learning models.

By the end of this notebook you will understand:
- Why training accuracy is misleading
- How train/test split works and why one split is unstable
- How model complexity causes overfitting
- Why validation sets are needed
- How to implement cross-validation (manually and with sklearn)

**Key protocol**: Train -> Validate / Cross-Validate -> Test

This notebook prepares you for **Week 8: Hyperparameter Tuning and AutoML**.

## Section 0: Setup

Libraries we'll use:
- `numpy` / `pandas` -- data manipulation
- `matplotlib` -- plotting
- `sklearn.model_selection` -- train/test splits and cross-validation
- `sklearn.tree` / `sklearn.linear_model` -- models
- `sklearn.preprocessing` / `sklearn.pipeline` -- data transformations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold,
)

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All imports successful!')

## Section 1: Create a Custom Dataset (Classification)

We'll create a **student performance dataset**. Each row is a student with four features:

| Feature | Description | Range |
|---------|-------------|-------|
| `study_hours` | Hours studied per week | 0 -- 10 |
| `attendance` | Class attendance percentage | 50 -- 100 |
| `sleep_hours` | Average sleep per night | 4 -- 9 |
| `previous_score` | Score in prerequisite course | 40 -- 95 |

**Target**: `pass_fail` (1 = passed, 0 = failed)

The pass probability depends on a combination of these features, with some randomness.

In [ ]:
np.random.seed(42)
n_students = 500

# Generate features
study_hours = np.random.uniform(0, 10, n_students)
attendance = np.random.uniform(50, 100, n_students)
sleep_hours = np.random.uniform(4, 9, n_students)
previous_score = np.random.uniform(40, 95, n_students)

# Compute pass probability using a logistic function
# More study, higher attendance, better sleep, higher prev score -> more likely to pass
z = (
    0.4 * study_hours
    + 0.03 * attendance
    + 0.3 * sleep_hours
    + 0.02 * previous_score
    - 5.0  # offset to center the probabilities
)

# Sigmoid function converts z to a probability between 0 and 1
prob_pass = 1 / (1 + np.exp(-z))

# Sample pass/fail from these probabilities (adds randomness)
pass_fail = (np.random.random(n_students) < prob_pass).astype(int)

# Create a DataFrame for easy viewing
df = pd.DataFrame({
    'study_hours': study_hours,
    'attendance': attendance,
    'sleep_hours': sleep_hours,
    'previous_score': previous_score,
    'pass_fail': pass_fail
})

print(f"Dataset: {len(df)} students")
print(f"Pass rate: {df['pass_fail'].mean():.1%}")

In [ ]:
# Quick look at the data
df.head(10)

In [ ]:
df.describe().round(2)

In [ ]:
# Visualize: study_hours vs pass_fail
fig, ax = plt.subplots(figsize=(8, 4))

passed = df[df['pass_fail'] == 1]
failed = df[df['pass_fail'] == 0]

ax.scatter(failed['study_hours'], failed['attendance'],
           c='#C44E52', alpha=0.5, s=20, label='Failed')
ax.scatter(passed['study_hours'], passed['attendance'],
           c='#4C72B0', alpha=0.5, s=20, label='Passed')

ax.set_xlabel('Study Hours / Week', fontsize=12)
ax.set_ylabel('Attendance %', fontsize=12)
ax.set_title('Student Performance Dataset', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Students who study more and attend class more tend to pass.")
print("But there is overlap -- the boundary is not clean.")

In [ ]:
# Prepare features (X) and target (y) as numpy arrays
X = df[['study_hours', 'attendance', 'sleep_hours', 'previous_score']].values
y = df['pass_fail'].values

print(f"Features shape: {X.shape}  (500 students, 4 features)")
print(f"Target shape:   {y.shape}")

## Section 2: Why Training Accuracy Is Misleading

A common beginner mistake: train a model on the entire dataset, then check its accuracy on that same dataset.

Let's see why this gives a false sense of performance.

In [ ]:
# Train a decision tree on ALL the data (no depth limit)
model = DecisionTreeClassifier(random_state=42)  # unlimited depth
model.fit(X, y)

train_score = model.score(X, y)
print(f"Training accuracy: {train_score:.1%}")

**100% accuracy!** Should we celebrate?

**No.** The model memorized every single training example. It created a specific rule for each student.

This is like a student who memorizes all practice problems word-for-word:
- Score on practice: 100%
- Score on the actual exam: much lower

**Training accuracy tells you how well the model MEMORIZES, not how well it GENERALIZES.**

We need to evaluate on data the model has never seen.

## Section 3: Train/Test Split

**Idea**: Hold out some data that the model never sees during training. Evaluate on that held-out data.

Typical split: 80% train, 20% test.

In [ ]:
# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

In [ ]:
# Train a decision tree (no depth limit) on the TRAINING set only
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate on BOTH sets
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print(f"Training accuracy: {train_acc:.1%}")
print(f"Test accuracy:     {test_acc:.1%}")
print(f"Gap:               {train_acc - test_acc:.1%}")

**Observations**:
- Training accuracy is very high (the model memorized the training data)
- Test accuracy is much lower (on new data, the memorized rules don't work)
- The **gap** between train and test accuracy = **overfitting indicator**

The test accuracy is our actual estimate of real-world performance.

But can we trust this single number? Let's find out.

## Section 4: Instability of a Single Split

The test accuracy depends on **which** students happened to end up in the test set. Different random splits give different results.

Let's run 30 different splits and see how much the score varies.

In [ ]:
# Try 30 different random splits
scores = []

for seed in range(30):
    # Each seed gives a different random split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_tr, y_tr)
    scores.append(model.score(X_te, y_te))

scores = np.array(scores)

print(f"Min test accuracy:  {scores.min():.1%}")
print(f"Max test accuracy:  {scores.max():.1%}")
print(f"Range:              {scores.max() - scores.min():.1%}")
print(f"Mean:               {scores.mean():.1%}")
print(f"Std:                {scores.std():.1%}")

In [ ]:
# Plot the distribution of test accuracies
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(scores, bins=12, color='#4C72B0', edgecolor='black', alpha=0.8)
ax.axvline(scores.mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean = {scores.mean():.1%}')

ax.set_xlabel('Test Accuracy', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('30 Random Train/Test Splits: Accuracy Varies Widely', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Key takeaway**: One split gives a noisy estimate. The accuracy can vary by many percentage points just depending on the split.

We need a more stable method. That's **cross-validation** (Section 7).

But first, let's understand WHY overfitting happens -- model complexity.

## Section 5: Model Complexity and Overfitting

Every model has a "complexity knob". Turning it up makes the model more flexible, but too much flexibility means it memorizes noise instead of learning patterns.

### 5a. Regression Example

Let's create a dataset where the true relationship is a smooth curve. We'll fit polynomials of increasing degree and watch what happens.

In [ ]:
# Create a nonlinear regression dataset
# Think of this as: temperature -> electricity usage
np.random.seed(42)
n_points = 40

x = np.sort(np.random.uniform(0, 10, n_points))
y_true = np.sin(1.5 * x) * 3 + 0.5 * x  # the true underlying pattern
noise = np.random.normal(0, 1.5, n_points)
y_noisy = y_true + noise

# Reshape for sklearn (needs 2D input)
X_reg = x.reshape(-1, 1)
y_reg = y_noisy

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x, y_noisy, c='gray', s=40, edgecolors='black',
           linewidths=0.5, label='Data (noisy)')
x_smooth = np.linspace(0, 10, 200)
ax.plot(x_smooth, np.sin(1.5 * x_smooth) * 3 + 0.5 * x_smooth,
        'g--', alpha=0.5, label='True function (hidden)')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Regression Dataset: True Pattern + Noise', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("The model only sees the gray dots. It does NOT know the green dashed line.")
print("Can it learn the pattern without memorizing the noise?")

In [ ]:
# Fit polynomials of degree 1, 3, and 15
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 3, 15]
titles = ['Degree 1 (Underfitting)', 'Degree 3 (Good Fit)', 'Degree 15 (Overfitting)']
colors = ['#C44E52', '#55A868', '#4C72B0']

x_plot = np.linspace(0, 10, 200).reshape(-1, 1)

for ax, deg, title, color in zip(axes, degrees, titles, colors):
    # Build a polynomial regression pipeline
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_reg, y_reg)
    y_pred = pipe.predict(x_plot)

    # Plot
    ax.scatter(x, y_noisy, c='gray', s=30, edgecolors='black', linewidths=0.5)
    ax.plot(x_plot, y_pred, color=color, linewidth=2.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_ylim(-10, 15)

plt.tight_layout()
plt.show()

print("Degree 1:  Too simple -- misses the curve entirely (UNDERFITTING)")
print("Degree 3:  Captures the pattern without memorizing noise (GOOD FIT)")
print("Degree 15: Wiggles through every data point -- memorizes noise (OVERFITTING)")

### 5b. Training Error vs Test Error as Complexity Increases

Let's quantify this: how do training and test scores change as polynomial degree increases?

In [ ]:
# Split regression data into train/test
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# Track train/test R-squared for degrees 1 through 15
degrees_range = range(1, 16)
train_r2 = []
test_r2 = []

for deg in degrees_range:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_r_train, y_r_train)
    train_r2.append(pipe.score(X_r_train, y_r_train))
    test_r2.append(pipe.score(X_r_test, y_r_test))

# Plot both curves
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(degrees_range), train_r2, 'b-o', linewidth=2, markersize=5,
        label='Training R-squared')
ax.plot(list(degrees_range), test_r2, 'r-o', linewidth=2, markersize=5,
        label='Test R-squared')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=12)
ax.set_ylabel('R-squared', fontsize=12)
ax.set_title('Training Error Always Decreases, But Test Error May Increase', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(list(degrees_range))
plt.tight_layout()
plt.show()

best_deg = list(degrees_range)[np.argmax(test_r2)]
print(f"Key lesson:")
print(f"  Training R-squared always goes up (more complex = better memorization)")
print(f"  Test R-squared peaks at degree {best_deg} then drops (overfitting!)")

### 5c. Decision Tree Depth: Same Story for Classification

Back to our student dataset. What happens as we increase tree depth?

In [ ]:
# Track train/test accuracy for different tree depths
depths = [1, 2, 3, 5, 7, 10, 15, 20, None]  # None = unlimited
train_accs = []
test_accs = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(dt.score(X_train, y_train))
    test_accs.append(dt.score(X_test, y_test))

# Print as a table
print(f"{'Depth':>6s}  {'Train Acc':>10s}  {'Test Acc':>10s}  {'Gap':>6s}")
print('=' * 38)
for d, tr, te in zip(depths, train_accs, test_accs):
    d_str = str(d) if d is not None else 'None'
    print(f"{d_str:>6s}  {tr:>10.1%}  {te:>10.1%}  {tr - te:>6.1%}")

In [ ]:
# Plot train vs test accuracy
fig, ax = plt.subplots(figsize=(9, 5))

x_labels = [str(d) if d is not None else 'None' for d in depths]
x_pos = range(len(depths))

ax.plot(x_pos, train_accs, 'b-o', linewidth=2, markersize=6, label='Training Accuracy')
ax.plot(x_pos, test_accs, 'r-o', linewidth=2, markersize=6, label='Test Accuracy')

ax.set_xticks(list(x_pos))
ax.set_xticklabels(x_labels)
ax.set_xlabel('Max Tree Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Decision Tree: Deeper = More Overfitting', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Notice:")
print("  Training accuracy reaches 100% with unlimited depth")
print("  Test accuracy peaks and then drops")
print("  The GAP between train and test = overfitting indicator")
print(f"\nQuestion: which depth is best? We need a VALIDATION SET to decide.")

## Section 6: Validation Set

We need to *choose* the best tree depth. But:
- Can't use training accuracy (always prefers more complex)
- Can't use test accuracy (contaminates final evaluation)

**Solution**: Split the data three ways.

```
Train (60%)      -> model learns parameters
Validation (20%) -> choose best hyperparameters
Test (20%)       -> final one-time evaluation (sealed envelope)
```

In [ ]:
# Three-way split: 60% train, 20% validation, 20% test
X_trainval, X_test_final, y_trainval, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_v, X_val, y_train_v, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42  # 0.25 of 80% = 20%
)

print(f"Training:   {len(X_train_v)} samples (60%)")
print(f"Validation: {len(X_val)} samples (20%)")
print(f"Test:       {len(X_test_final)} samples (20%)")

In [ ]:
# Try different tree depths, evaluate on the VALIDATION set
depths_to_try = [1, 2, 3, 5, 7, 10, 15, 20]
val_scores = []

print(f"{'Depth':>6s}  {'Train Acc':>10s}  {'Val Acc':>10s}")
print('=' * 30)

for depth in depths_to_try:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_v, y_train_v)
    train_acc = dt.score(X_train_v, y_train_v)
    val_acc = dt.score(X_val, y_val)
    val_scores.append(val_acc)
    print(f"{depth:>6d}  {train_acc:>10.1%}  {val_acc:>10.1%}")

# Find the best depth
best_depth = depths_to_try[np.argmax(val_scores)]
print(f"\nBest depth (by validation): {best_depth}")

In [ ]:
# Step 1: Retrain best model on ALL training + validation data
best_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_model.fit(X_trainval, y_trainval)  # uses 80% of data

# Step 2: Evaluate ONCE on the test set
final_test_acc = best_model.score(X_test_final, y_test_final)

print(f"Best depth: {best_depth}")
print(f"Final test accuracy: {final_test_acc:.1%}")
print()
print("This is our HONEST estimate of real-world performance.")
print("We used the test set exactly ONCE -- no contamination.")

**Problem with this approach**: We're only training on 60% of the data. With small datasets, this hurts model quality.

Also, the validation score depends on *which* 20% of students ended up in the validation set -- same variance problem as before!

**Solution**: Cross-validation -- use ALL data for both training and validation.

## Section 7: Manual Cross-Validation

**K-Fold Cross-Validation**:
1. Split the data into K equal folds
2. For each fold: use it as validation, train on the remaining K-1 folds
3. Average all K scores

$$\text{CV Score} = \frac{1}{K} \sum_{k=1}^{K} \text{score}_k$$

Let's implement this **from scratch** before using sklearn's shortcut.

In [ ]:
# Step 1: Shuffle the data and split into K folds
K = 5

indices = np.arange(len(X))
np.random.seed(42)
np.random.shuffle(indices)

# Split into K roughly equal-sized groups
folds = np.array_split(indices, K)

print(f"Total samples: {len(X)}")
print(f"Number of folds: {K}")
print(f"Fold sizes: {[len(f) for f in folds]}")

In [ ]:
# Step 2: For each fold, train on K-1 folds, evaluate on the held-out fold
scores_manual = []

for k in range(K):
    # This fold is held out for validation
    val_idx = folds[k]
    
    # All other folds are used for training
    train_idx = np.concatenate([folds[j] for j in range(K) if j != k])

    # Split the data
    X_train_cv = X[train_idx]
    y_train_cv = y[train_idx]
    X_val_cv = X[val_idx]
    y_val_cv = y[val_idx]

    # Train and evaluate
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X_train_cv, y_train_cv)
    score = model.score(X_val_cv, y_val_cv)
    scores_manual.append(score)

    print(f"Fold {k+1}: trained on {len(train_idx)} samples, "
          f"validated on {len(val_idx)} samples -> accuracy = {score:.3f}")

# Step 3: Average the scores
print(f"\nCV Score: {np.mean(scores_manual):.3f} +/- {np.std(scores_manual):.3f}")

**What just happened:**
- Each student was in the validation set **exactly once**
- Each student was in the training set **4 out of 5 times**
- We got 5 accuracy estimates and averaged them
- The standard deviation tells us how stable the estimate is

This is much more reliable than a single train/test split!

## Section 8: sklearn Cross-Validation

Everything we just did manually can be done in **two lines** with sklearn's `cross_val_score`.

In [ ]:
# The sklearn way: two lines!
model = DecisionTreeClassifier(max_depth=5, random_state=42)
scores = cross_val_score(model, X, y, cv=5)

print(f"Fold scores: {scores}")
print(f"Mean:        {scores.mean():.3f}")
print(f"Std:         {scores.std():.3f}")
print(f"\nReport as: {scores.mean():.1%} +/- {scores.std():.1%} (5-fold CV)")

**Advantages of `cross_val_score`**:
- Simpler code (2 lines vs 15)
- Less error-prone (no manual index shuffling)
- Standard practice in ML
- Handles stratification automatically for classifiers

Let's use it to find the best tree depth.

In [ ]:
# Use CV to find the best tree depth
depths_to_try = [1, 2, 3, 5, 7, 10, 15, 20]
cv_means = []
cv_stds = []

print(f"{'Depth':>6s}  {'CV Mean':>10s}  {'CV Std':>10s}")
print('=' * 30)

for depth in depths_to_try:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(model, X, y, cv=5)
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f"{depth:>6d}  {scores.mean():>10.3f}  {scores.std():>10.3f}")

best_cv_depth = depths_to_try[np.argmax(cv_means)]
print(f"\nBest depth by CV: {best_cv_depth} (CV = {max(cv_means):.3f})")

In [ ]:
# Plot CV scores with error bars
fig, ax = plt.subplots(figsize=(9, 5))

ax.errorbar(depths_to_try, cv_means, yerr=cv_stds,
            fmt='o-', linewidth=2, markersize=6, capsize=4,
            color='#4C72B0', ecolor='gray')

ax.set_xlabel('Max Tree Depth', fontsize=12)
ax.set_ylabel('CV Accuracy (mean +/- std)', fontsize=12)
ax.set_title('Cross-Validation Scores for Different Tree Depths', fontsize=13)
ax.set_xticks(depths_to_try)
plt.tight_layout()
plt.show()

print(f"Best depth: {best_cv_depth}")
print(f"The error bars show the uncertainty in each estimate.")
print(f"Overlapping error bars = not significantly different.")

## Section 9: Stratified Cross-Validation

**Problem**: If the dataset is imbalanced (e.g., 90% pass, 10% fail), a random fold might accidentally get 100% pass students and 0% fail students. The model would get misleading scores.

**Stratified K-Fold** ensures each fold has the **same class ratio** as the full dataset.

In [ ]:
# Check our class distribution
print(f"Overall pass rate: {y.mean():.1%}")
print()

# Compare regular KFold vs StratifiedKFold
print("=== Regular KFold ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_num, (train_idx, val_idx) in enumerate(kf.split(X)):
    fold_pass_rate = y[val_idx].mean()
    print(f"  Fold {fold_num+1}: pass rate = {fold_pass_rate:.1%}")

print()
print("=== Stratified KFold ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_num, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    fold_pass_rate = y[val_idx].mean()
    print(f"  Fold {fold_num+1}: pass rate = {fold_pass_rate:.1%}")

In [ ]:
# Use StratifiedKFold with cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = DecisionTreeClassifier(max_depth=5, random_state=42)
scores_stratified = cross_val_score(model, X, y, cv=skf)

print(f"Stratified CV scores: {scores_stratified}")
print(f"Mean: {scores_stratified.mean():.3f} +/- {scores_stratified.std():.3f}")
print()
print("Good news: cross_val_score uses StratifiedKFold by DEFAULT for classifiers!")
print("So 'cross_val_score(model, X, y, cv=5)' already does stratification.")

## Section 10: The Correct Evaluation Protocol

Let's put everything together in one clean workflow.

In [ ]:
# =========================================
# THE CORRECT EVALUATION PROTOCOL
# =========================================

# STEP 1: Split off a test set (20%). Lock it away.
X_dev, X_test_final, y_dev, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Step 1: Held out {len(X_test_final)} test samples (LOCKED AWAY)")
print(f"        Remaining: {len(X_dev)} dev samples for CV")
print()

In [ ]:
# STEP 2: Use CV on dev set to compare models
print("Step 2: Compare models using 5-fold CV on dev set")
print()

models_to_try = [
    ("LogReg", LogisticRegression(max_iter=1000)),
    ("Tree (d=3)", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("Tree (d=5)", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("Tree (d=10)", DecisionTreeClassifier(max_depth=10, random_state=42)),
]

best_name, best_score, best_model_obj = None, 0, None

print(f"{'Model':>15s}  {'CV Mean':>10s}  {'CV Std':>10s}")
print('=' * 40)
for name, model in models_to_try:
    cv_scores = cross_val_score(model, X_dev, y_dev, cv=5)
    print(f"{name:>15s}  {cv_scores.mean():>10.3f}  {cv_scores.std():>10.3f}")
    if cv_scores.mean() > best_score:
        best_name = name
        best_score = cv_scores.mean()
        best_model_obj = model

print(f"\nStep 3: Best model = {best_name} (CV = {best_score:.3f})")

In [ ]:
# STEP 4: Train best model on ALL dev data
print(f"Step 4: Train {best_name} on all {len(X_dev)} dev samples")
best_model_obj.fit(X_dev, y_dev)
print()

In [ ]:
# STEP 5: Evaluate ONCE on the test set
final_acc = best_model_obj.score(X_test_final, y_test_final)
print(f"Step 5: Final test accuracy = {final_acc:.1%}")
print()
print("This is our HONEST, UNBIASED estimate of real-world performance.")
print("The test set was used exactly ONCE.")

## Summary

The correct evaluation protocol:

```
1. Split off a test set first (20%). Lock it away.
2. Use cross-validation on remaining data for model selection.
3. Choose the best model / hyperparameters.
4. Retrain on ALL non-test data.
5. Evaluate ONCE on the test set. Report this number.
```

**Do NOT tune hyperparameters on the test set.**

| What we learned | Key takeaway |
|---|---|
| Training accuracy | Measures memorization, not generalization |
| Train/test split | Better, but one split is unstable |
| Model complexity | More complex != better (overfitting) |
| Validation set | Separate model selection from final evaluation |
| K-fold CV | Use all data for both training and validation |
| Stratified CV | Maintain class ratios in each fold |

**Next week**: We'll use cross-validation *inside* automated tuning methods:
- Grid Search
- Random Search
- Bayesian Optimization
- AutoML
- Experiment Tracking